# Vesuvius Scroll Segment Visualizer

Explore the 7 downloaded PHercParis4 scroll segments from `data/scroll` (65-layer Z-stacks, **no labels**).

```
data/scroll/{segment_id}/
  surface_volume/00–64.tif   ← 65 z-slices
  mask.png                   ← valid papyrus region
```

**Sections:** mask · interactive Z-browser · orthogonal projections · depth statistics · cross-section · side-by-side comparison

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path
from PIL import Image
import tifffile as tiff
import ipywidgets as widgets
from IPython.display import display

DATA_ROOT = Path("../data/scroll")
SEGMENTS  = sorted([p.name for p in DATA_ROOT.iterdir() if p.is_dir()])

def load_slice(seg_dir, z):
    return tiff.imread(str(seg_dir / "surface_volume" / f"{z:02d}.tif")).astype(np.float32)

def load_mask(seg_dir):
    return np.array(Image.open(str(seg_dir / "mask.png")).convert("L")) > 128

def normalize(arr, mask=None):
    src = arr[mask] if mask is not None else arr.ravel()
    p1, p99 = np.percentile(src, [1, 99])
    return np.clip((arr - p1) / (p99 - p1 + 1e-8), 0, 1)

print(f"Found {len(SEGMENTS)} segments:")
for s in SEGMENTS:
    n = len(list((DATA_ROOT / s / "surface_volume").glob("*.tif")))
    print(f"  {s}  ({n} layers)")

seg_sel = widgets.Dropdown(options=SEGMENTS, description="Segment:")
display(seg_sel)

## 1. Load Segment

Re-run this cell after changing the dropdown to switch segments.

In [ ]:
SEG_ID  = seg_sel.value
seg_dir = DATA_ROOT / SEG_ID
tifs     = sorted((seg_dir / "surface_volume").glob("*.tif"))
num_layers = len(tifs)

mask = load_mask(seg_dir)

print(f"Segment : {SEG_ID}")
print(f"Layers  : {num_layers}")
print(f"Shape   : {mask.shape}")
print(f"Valid px: {mask.sum():,}  ({mask.sum()/mask.size*100:.1f}%)")

fig, ax = plt.subplots(figsize=(6, 6))
ax.imshow(mask, cmap="gray")
ax.set_title(f"{SEG_ID} — papyrus mask"); ax.axis("off")
plt.tight_layout(); plt.show()

## 2. Interactive Z-Stack Browser

In [ ]:
@widgets.interact(z=widgets.IntSlider(min=0, max=num_layers-1, value=num_layers//2, description="Layer"))
def view_layer(z=num_layers//2):
    sl   = load_slice(seg_dir, z)
    disp = normalize(sl, mask)
    masked = disp.copy(); masked[~mask] = 0

    fig, axes = plt.subplots(1, 3, figsize=(16, 5))
    fig.suptitle(f"{SEG_ID} — Layer z={z:02d}", fontsize=12)

    axes[0].imshow(disp, cmap="gray")   ; axes[0].set_title("CT slice")                ; axes[0].axis("off")
    axes[1].imshow(masked, cmap="gray") ; axes[1].set_title("Masked (valid papyrus)")  ; axes[1].axis("off")
    axes[2].hist(sl[mask].ravel(), bins=200, color="steelblue", alpha=0.8)
    axes[2].set_title("Intensity histogram"); axes[2].set_xlabel("Raw pixel value")

    plt.tight_layout(); plt.show()

## 3. Orthogonal Projections (MIP + Mean)

Builds a downsampled 3-D volume first. Increase `DOWNSAMPLE` if you run out of RAM.

In [ ]:
DOWNSAMPLE = 4
print(f"Loading volume (spatial ×{DOWNSAMPLE} downsample, all {num_layers} layers)...")
vol = np.stack(
    [load_slice(seg_dir, z)[::DOWNSAMPLE, ::DOWNSAMPLE] for z in range(num_layers)],
    axis=0
)
print(f"Volume: {vol.shape}  ({vol.nbytes/1e6:.0f} MB)")

fig, axes = plt.subplots(2, 3, figsize=(15, 9))
views = [
    ("MIP — axial (Z→)",    vol.max(0)),
    ("MIP — coronal (Y→)",  vol.max(1)),
    ("MIP — sagittal (X→)", vol.max(2)),
    ("Mean — axial",         vol.mean(0)),
    ("Mean — coronal",        vol.mean(1)),
    ("Mean — sagittal",       vol.mean(2)),
]
for ax, (title, img) in zip(axes.ravel(), views):
    ax.imshow(img, cmap="gray"); ax.set_title(title); ax.axis("off")

plt.suptitle(f"{SEG_ID} — orthogonal projections (downsample={DOWNSAMPLE})", fontsize=13)
plt.tight_layout(); plt.show()

## 4. Depth Statistics

Per-layer mean/std across all pixels inside the valid mask.

In [ ]:
print("Computing per-layer statistics...")
means, stds, mins, maxs = [], [], [], []
for z in range(num_layers):
    sl = load_slice(seg_dir, z)
    v  = sl[mask]
    means.append(v.mean()); stds.append(v.std())
    mins.append(v.min());   maxs.append(v.max())
means, stds, mins, maxs = map(np.array, [means, stds, mins, maxs])
zs = np.arange(num_layers)

fig, axes = plt.subplots(1, 3, figsize=(16, 4))

axes[0].plot(zs, means, lw=1.5)
axes[0].fill_between(zs, means - stds, means + stds, alpha=0.2, label="±1 std")
axes[0].set_title("Mean ± std per layer"); axes[0].set_xlabel("Z layer"); axes[0].legend()
axes[0].grid(True, ls="--", alpha=0.4)

axes[1].fill_between(zs, mins, maxs, alpha=0.25, color="green", label="min–max")
axes[1].plot(zs, mins, color="green", lw=1); axes[1].plot(zs, maxs, color="green", lw=1)
axes[1].set_title("Min / Max per layer"); axes[1].set_xlabel("Z layer"); axes[1].legend()
axes[1].grid(True, ls="--", alpha=0.4)

axes[2].hist(means, bins=20, color="steelblue", edgecolor="white")
axes[2].set_title("Distribution of layer means"); axes[2].set_xlabel("Mean intensity")

plt.suptitle(f"{SEG_ID} — layer statistics", fontsize=12)
plt.tight_layout(); plt.show()

## 5. Cross-Section View

Vertical slice through the Z-stack — the bright band is the papyrus surface.

In [ ]:
valid_rows = np.where(mask.any(axis=1))[0]
row = valid_rows[len(valid_rows) // 2]

cross = np.stack([load_slice(seg_dir, z)[row] for z in range(num_layers)], axis=0)

fig, ax = plt.subplots(figsize=(16, 4))
ax.imshow(normalize(cross), cmap="gray", aspect="auto")
ax.axhline(num_layers // 2, color="red", ls="--", lw=0.9, label=f"z={num_layers//2} (surface)")
ax.set_xlabel("X position"); ax.set_ylabel("Z layer (0=above, {num_layers-1}=below)")
ax.set_title(f"Cross-section at row {row} — {SEG_ID}")
ax.legend()
plt.tight_layout(); plt.show()

## 6. Side-by-Side Comparison — All Segments

Middle layer of every downloaded segment.

In [ ]:
Z_COMPARE = 32
ncols = min(4, len(SEGMENTS))
nrows = (len(SEGMENTS) + ncols - 1) // ncols

fig, axes = plt.subplots(nrows, ncols, figsize=(4 * ncols, 4 * nrows))
axes = np.array(axes).ravel()

for ax, seg in zip(axes, SEGMENTS):
    sd  = DATA_ROOT / seg
    sv  = sd / "surface_volume"
    # Use available layer count — some segments have fewer layers
    n   = len(list(sv.glob("*.tif")))
    z   = min(Z_COMPARE, n - 1)
    sl  = load_slice(sd, z)
    mk  = load_mask(sd)
    ax.imshow(normalize(sl, mk), cmap="gray")
    ax.set_title(seg, fontsize=7); ax.axis("off")

for ax in axes[len(SEGMENTS):]:
    ax.axis("off")

plt.suptitle(f"All segments — layer z={Z_COMPARE}", fontsize=13)
plt.tight_layout(); plt.show()

## 7. Layer Filmstrip

All Z-layers for the selected segment as thumbnails.

In [ ]:
THUMB = 128
NCOLS = 13
nrows_film = (num_layers + NCOLS - 1) // NCOLS

fig, axes = plt.subplots(nrows_film, NCOLS, figsize=(NCOLS * 2, nrows_film * 2))
fig.suptitle(f"All {num_layers} Z-layers — {SEG_ID}", fontsize=12)
axes = np.array(axes).ravel()

for z in range(num_layers):
    sl   = load_slice(seg_dir, z)
    disp = normalize(sl, mask)
    thumb = np.array(Image.fromarray((disp * 255).astype(np.uint8)).resize((THUMB, THUMB)))
    axes[z].imshow(thumb, cmap="gray")
    axes[z].set_title(f"z={z:02d}", fontsize=6)
    axes[z].axis("off")

for ax in axes[num_layers:]:
    ax.axis("off")

plt.tight_layout(); plt.show()